In [17]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments,T5ForConditionalGeneration

In [18]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [19]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [21]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [22]:
train_data.sample(5)

,id,dialogue,summary
9338,13862854,Martin: did you see the new episode of Grand T...,Martin thinks that Grand Tour is now way bette...
11761,13828453,Dean: I feel sick\r\nScott: hungover?\r\nDean:...,Dean is feeling sick and the reason can be Chi...
1719,13814330,Zoe: uncle Stephen died\r\nRyan: do you know w...,Uncle Stephen Died of heart attack. Grandpa is...
5573,13730121,Megan: Have you heard of the shooting on State...,In the shooting on Staten Island a policeman w...
3218,13717018,Jason: Although he is not a good striker with ...,Jason reckons the Arsenal player is not a good...


In [23]:
train_data.shape

(14732, 3)

In [24]:
val_data.shape

(818, 3)

In [25]:
#random samply
train_data = train_data.sample(n=4000,random_state=42).reset_index(drop = True)
val_data = val_data.sample(n=500,random_state=42).reset_index(drop=True)

# Pre Processing

In [26]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text) #removing lines
    text = re.sub(r"\s+"," ",text) #spaces
    text = re.sub(r"<.*?>","",text) #html tags
    text = text.strip().lower()
    return text

In [27]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenization

In [28]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [29]:
#raw data ==> tokenized input for fine-tuning
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation = True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"] #token ids ==> add to input as labels
    return inputs

In [30]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()


In [31]:
train_dataset[0]

{'input_ids': [25208, 10, 107, 23, 55, 2617, 526, 9, 11465, 8048, 2064, 17, 77, 31, 7, 8372, 232, 23, 11841, 17, 6279, 4188, 51, 2632, 8954, 155, 19405, 53, 11275, 15, 17, 10, 7997, 15, 10, 107, 23, 55, 10, 61, 20349, 7, 6, 2780, 23, 31, 162, 138, 23015, 5236, 155, 5, 10, 61, 7997, 15, 10, 2780, 20349, 7, 1161, 10337, 53, 7932, 526, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [16]:
#input ids - DIALOGUE => tokenids
#attention mask
#labels - target ==> summary

# Working with model

In [32]:
#NLP => Generation based task
model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights: 100%|██████████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 8511.67it/s]


In [37]:
#Fine Tune
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else: 
    device = torch.device("cpu")

print("device is:",device)
model.to(device)

device is: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [40]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs  = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    warmup_steps = 500
)

In [42]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

# Training dataset

In [43]:
trainer.train()

/home/saphal/anaconda3/envs/ml_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,3.613889,0.701805
2,0.710093,0.622723
3,0.658805,0.601893
4,0.634594,0.590060
5,0.621268,0.585130
6,0.614711,0.582052


Writing model shards: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.77it/s]
/home/saphal/anaconda3/envs/ml_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.63it/s]
/home/saphal/anaconda3/envs/ml_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.62it/s]
/home/saphal/anaconda3/envs/ml_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argumen

TrainOutput(global_step=3000, training_loss=1.142226806640625, metrics={'train_runtime': 38411.114, 'train_samples_per_second': 0.625, 'train_steps_per_second': 0.078, 'total_flos': 3248203235328000.0, 'train_loss': 1.142226806640625, 'epoch': 6.0})

In [44]:
# model load ==> fine - tune ==> save the model

In [45]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.07it/s]


('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [47]:
#use the model and tokenizer
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|█████████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 10660.31it/s]


## Test the core logic


In [50]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) #clean dialogue

    #tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    )

    #generate summary ( token ids)
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams=4 ,# transformer compare 4 output to give best summary
        early_stopping = True

    )

    #token ids to summary ==> decoding
    summary = tokenizer.decode(targets[0],skip_special_tokens=True) #EOS, SEP
    return summary


In [51]:
#sample 
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summary = summarize_dialogue(test_dialogue)

print("Summary:",summary)

Summary: reporterandexpertareinvestingheavilyinmachinelearningsystemstoautomatetasks,improvedecision-making,andenhancecustomerexperiences.aisystemsarebecomingmorecapableduetoadvancesindeeplearningandaccesstolargedatasets.
